In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/ML/datasets ML4/training.csv', sep=',', parse_dates=['PurchDate'])
data.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,2009-12-07,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,2009-12-07,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,2009-12-07,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,2009-12-07,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,2009-12-07,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   RefId                              72983 non-null  int64         
 1   IsBadBuy                           72983 non-null  int64         
 2   PurchDate                          72983 non-null  datetime64[ns]
 3   Auction                            72983 non-null  object        
 4   VehYear                            72983 non-null  int64         
 5   VehicleAge                         72983 non-null  int64         
 6   Make                               72983 non-null  object        
 7   Model                              72983 non-null  object        
 8   Trim                               70623 non-null  object        
 9   SubModel                           72975 non-null  object        
 10  Color                             

бесполезными являются фичи: PurchDate, Color, RefId, BYRNO, PRIMEUNIT, AUCGUART

создают зависимость колонки: VehYear, VNZIP1, TopThreeAmericanName, MMR..., VehBCost

In [ ]:
data = data.drop(columns=['Color', 'RefId', 'BYRNO', 'PRIMEUNIT', 'AUCGUART', 'VehYear', 'VNZIP1', 'TopThreeAmericanName', 'WheelTypeID'])

In [ ]:
data['PriceSpread'] = data['MMRAcquisitionAuctionAveragePrice'] - data['MMRAcquisitionRetailAveragePrice']
data['PriceDrop'] = data['MMRAcquisitionAuctionAveragePrice'] - data['MMRCurrentAuctionAveragePrice']

In [ ]:
cols_to_drop = [
    'MMRAcquisitionAuctionCleanPrice',
    'MMRAcquisitionRetailAveragePrice',
    'MMRAcquisitonRetailCleanPrice',
    'MMRCurrentAuctionAveragePrice',
    'MMRCurrentAuctionCleanPrice',
    'MMRCurrentRetailAveragePrice',
    'MMRCurrentRetailCleanPrice'
]

data = data.drop(columns=cols_to_drop)

в one-hot: Auction, Make, Transmission, wheelTypeID, Nationality, Size, VNST

в Target Encoding: Model, SubModel, Trim

In [ ]:
import numpy as np

data = data.sort_values(by=['PurchDate']).reset_index(drop=True)

unique_dates = np.sort(data['PurchDate'].unique())

n_dates = len(unique_dates)
third = n_dates // 3

train_dates = unique_dates[:third]
val_dates = unique_dates[third: 2 * third]
test_dates = unique_dates[2 * third:]

train = data[data['PurchDate'].isin(train_dates)].copy()
val = data[data['PurchDate'].isin(val_dates)].copy()
test = data[data['PurchDate'].isin(test_dates)].copy()

data = data.drop(columns=['PurchDate'])

train = train.drop(columns=['PurchDate'])
val = val.drop(columns=['PurchDate'])
test = test.drop(columns=['PurchDate'])

In [ ]:
make = train['Make'].value_counts()
common_marks = make[make >= 100].index

for df in [train, val, test]:
  df['Make'] = np.where(
    df['Make'].isin(common_marks),
    df['Make'],
    'OTHER'
  )

In [ ]:
missing = data.isna().sum()
cols_to_fill = missing[missing > 0].index
data[cols_to_fill].head()

,Trim,SubModel,Transmission,WheelType,Nationality,Size,MMRAcquisitionAuctionAveragePrice,PriceSpread,PriceDrop
0,Bas,4D SPORT,AUTO,Covers,AMERICAN,CROSSOVER,7261.0,-1081.0,-1448.0
1,SES,4D PASSENGER 3.9L SES,AUTO,Alloy,AMERICAN,VAN,4409.0,-853.0,-499.0
2,SE,4D SEDAN SE,AUTO,Covers,AMERICAN,MEDIUM,3098.0,-748.0,-299.0
3,LS,4D SUV 4.2L,AUTO,Alloy,AMERICAN,MEDIUM SUV,8530.0,-1182.0,-672.0
4,SES,4D SEDAN SES DURATEC,AUTO,Alloy,AMERICAN,MEDIUM,3094.0,-748.0,-275.0


In [ ]:
categorical_to_fill = ['Trim', 'SubModel', 'Transmission', 'WheelType', 'Nationality', 'Size']
data[categorical_to_fill] = data[categorical_to_fill].fillna('UNKNOWN')

In [ ]:
numeric_to_fill = ['MMRAcquisitionAuctionAveragePrice', 'PriceSpread', 'PriceDrop']

train_medians = train[numeric_to_fill].median()

train[numeric_to_fill] = train[numeric_to_fill].fillna(train_medians)
val[numeric_to_fill] = val[numeric_to_fill].fillna(train_medians)
test[numeric_to_fill] = test[numeric_to_fill].fillna(train_medians)

In [ ]:
X_train = train.drop(columns=['IsBadBuy'])
X_val = val.drop(columns=['IsBadBuy'])
X_test = test.drop(columns=['IsBadBuy'])

y_train = train['IsBadBuy']
y_val = val['IsBadBuy']
y_test = test['IsBadBuy']

In [ ]:
from sklearn.preprocessing import (
    TargetEncoder,
    OneHotEncoder
)

te = TargetEncoder(categories='auto', target_type='binary', smooth='auto')

target_cols = ['Model', 'SubModel', 'Trim']

train_values = te.fit_transform(X_train[target_cols], y_train)
val_values = te.transform(X_val[target_cols])
test_values = te.transform(X_test[target_cols])

X_train_encoded = X_train.copy()
X_val_encoded = X_val.copy()
X_test_encoded = X_test.copy()

X_train_encoded[target_cols] = train_values
X_val_encoded[target_cols] = val_values
X_test_encoded[target_cols] = test_values


ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

ohe_cols = ['Auction', 'Make', 'Transmission', 'WheelType', 'Nationality', 'Size', 'VNST']

train_values = ohe.fit_transform(X_train_encoded[ohe_cols])
val_values = ohe.transform(X_val_encoded[ohe_cols])
test_values = ohe.transform(X_test_encoded[ohe_cols])

train_ohe = pd.DataFrame(train_values,
                         columns=ohe.get_feature_names_out(ohe_cols),
                         index=X_train_encoded.index
                         )

val_ohe = pd.DataFrame(val_values,
                       columns=ohe.get_feature_names_out(ohe_cols),
                       index=X_val_encoded.index
                       )

test_ohe = pd.DataFrame(test_values,
                        columns=ohe.get_feature_names_out(ohe_cols),
                        index=X_test_encoded.index
                        )

X_train_final = pd.concat([X_train_encoded.drop(columns=ohe_cols), train_ohe], axis=1)
X_val_final = pd.concat([X_val_encoded.drop(columns=ohe_cols), val_ohe], axis=1)
X_test_final = pd.concat([X_test_encoded.drop(columns=ohe_cols), test_ohe], axis=1)

In [ ]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()

train_scaled = ss.fit_transform(X_train_final)
val_scaled = ss.transform(X_val_final)
test_scaled = ss.transform(X_test_final)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

models = {
    "logreg": LogisticRegression(C=0.01, max_iter=1000),
    "gnb": GaussianNB(),
    "knn": KNeighborsClassifier(n_neighbors=5)
}

def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1

for name, model in models.items():
    model.fit(train_scaled, y_train)

    proba = model.predict_proba(val_scaled)[:, 1]
    score = gini(y_val, proba)

    print(name, score)

logreg 0.41200345000948024
gnb 0.3315569139458767
knn 0.2730205610939329


In [ ]:
import numpy as np
import pandas as pd


def roc_auc_manual(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    n_pos = np.sum(y_true == 1)
    n_neg = np.sum(y_true == 0)

    ranks = pd.Series(y_score).rank(method='average').to_numpy()

    pos_ranks_sum = ranks[y_true == 1].sum()

    auc = (
        pos_ranks_sum
        - n_pos * (n_pos + 1) / 2
    ) / (n_pos * n_neg)

    return auc


def gini_manual(y_true, y_score):
    auc = roc_auc_manual(y_true, y_score)
    return 2 * auc - 1

In [ ]:
from sklearn.metrics import roc_auc_score

proba = model.predict_proba(val_scaled)[:, 1]

my_auc = roc_auc_manual(y_val, proba)
sk_auc = roc_auc_score(y_val, proba)

print("my auc:", my_auc)
print("sklearn auc:", sk_auc)

print("my gini:", gini_manual(y_val, proba))
print("sklearn gini:", abs(2 * sk_auc - 1))

my auc: 0.6365102805469665
sklearn auc: 0.6365102805469665
my gini: 0.2730205610939329
sklearn gini: 0.2730205610939329


In [ ]:
import numpy as np


class MyLogisticRegression:

    def __init__(
        self,
        lr=0.01,
        epochs=20,
        batch_size=64,
        random_state=42
    ):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.rng = np.random.default_rng(random_state)


    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))


    def fit(self, X, y):

        X = np.asarray(X)
        y = np.asarray(y)

        n_samples, n_features = X.shape

        self.w = np.zeros(n_features)
        self.b = 0.0

        for _ in range(self.epochs):

            idx = self.rng.permutation(n_samples)

            X_epoch = X[idx]
            y_epoch = y[idx]

            for start in range(0, n_samples, self.batch_size):

                end = start + self.batch_size

                xb = X_epoch[start:end]
                yb = y_epoch[start:end]

                logits = xb @ self.w + self.b
                probs = self._sigmoid(logits)

                error = probs - yb

                grad_w = xb.T @ error / len(xb)
                grad_b = error.mean()

                self.w -= self.lr * grad_w
                self.b -= self.lr * grad_b

        return self


    def predict_proba(self, X):

        X = np.asarray(X)

        probs = self._sigmoid(X @ self.w + self.b)

        return np.column_stack([
            1 - probs,
            probs
        ])


    def predict(self, X):
        return (
            self.predict_proba(X)[:, 1] >= 0.5
        ).astype(int)

In [ ]:
from sklearn.metrics import roc_auc_score

models = {
    "my_logreg": MyLogisticRegression()
}

for name, model in models.items():

    model.fit(train_scaled, y_train)

    proba = model.predict_proba(
        val_scaled
    )[:, 1]

    gini = 2 * roc_auc_score(
        y_val,
        proba
    ) - 1

    print(name, gini)

my_logreg 0.4576420307602691


In [ ]:
eps = 1e-6

for df in [train, val, test]:

    df['CostToMMR'] = (
        df['VehBCost']
        /
        (df['MMRAcquisitionAuctionAveragePrice'] + eps)
    )

    df['OdoPerYear'] = (
        df['VehOdo']
        /
        (df['VehicleAge'] + 1)
    )

    df['WarrantyPerCost'] = (
        df['WarrantyCost']
        /
        (df['VehBCost'] + eps)
    )

In [ ]:
make_mean = train.groupby(
    'Make'
)['VehBCost'].mean()

for df in [train, val, test]:

    df['MakeMeanCost'] = (
        df['Make']
        .map(make_mean)
    )

auction_mean = train.groupby(
    'Auction'
)['VehBCost'].mean()


model_mean = train.groupby(
    'Model'
)['VehBCost'].mean()

In [ ]:
X_train = train.drop(columns=['IsBadBuy'])
X_val = val.drop(columns=['IsBadBuy'])
X_test = test.drop(columns=['IsBadBuy'])

te = TargetEncoder(categories='auto', target_type='binary', smooth='auto')

target_cols = ['Model', 'SubModel', 'Trim']

train_values = te.fit_transform(X_train[target_cols], y_train)
val_values = te.transform(X_val[target_cols])
test_values = te.transform(X_test[target_cols])

X_train_encoded = X_train.copy()
X_val_encoded = X_val.copy()
X_test_encoded = X_test.copy()

X_train_encoded[target_cols] = train_values
X_val_encoded[target_cols] = val_values
X_test_encoded[target_cols] = test_values


ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

ohe_cols = ['Auction', 'Make', 'Transmission', 'WheelType', 'Nationality', 'Size', 'VNST']

train_values = ohe.fit_transform(X_train_encoded[ohe_cols])
val_values = ohe.transform(X_val_encoded[ohe_cols])
test_values = ohe.transform(X_test_encoded[ohe_cols])

train_ohe = pd.DataFrame(train_values,
                         columns=ohe.get_feature_names_out(ohe_cols),
                         index=X_train_encoded.index
                         )

val_ohe = pd.DataFrame(val_values,
                       columns=ohe.get_feature_names_out(ohe_cols),
                       index=X_val_encoded.index
                       )

test_ohe = pd.DataFrame(test_values,
                        columns=ohe.get_feature_names_out(ohe_cols),
                        index=X_test_encoded.index
                        )

X_train_final = pd.concat([X_train_encoded.drop(columns=ohe_cols), train_ohe], axis=1)
X_val_final = pd.concat([X_val_encoded.drop(columns=ohe_cols), val_ohe], axis=1)
X_test_final = pd.concat([X_test_encoded.drop(columns=ohe_cols), test_ohe], axis=1)

ss = StandardScaler()

train_scaled = ss.fit_transform(X_train_final)
val_scaled = ss.transform(X_val_final)
test_scaled = ss.transform(X_test_final)

In [ ]:
models = {
    "logreg": LogisticRegression(C=0.01, max_iter=1000),
    "gnb": GaussianNB(),
    "knn": KNeighborsClassifier(n_neighbors=5)
}

def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1

for name, model in models.items():
    model.fit(train_scaled, y_train)

    proba = model.predict_proba(val_scaled)[:, 1]
    score = gini(y_val, proba)

    print(name, score)

logreg 0.45787281411380754
gnb 0.3522660527859336
knn 0.2746272613575773


In [ ]:
coef = pd.Series(
    models['logreg'].coef_[0],
    index=X_train_final.columns
)

weak_features = coef[
    coef.abs() < 0.01
].index

X_train_manual = X_train_final.drop(columns=weak_features)
X_val_manual = X_val_final.drop(columns=weak_features)

ss_manual = StandardScaler()

X_train_manual_scaled = ss_manual.fit_transform(X_train_manual)
X_val_manual_scaled = ss_manual.transform(X_val_manual)

In [ ]:
model = LogisticRegression(
    C=0.01,
    max_iter=2000
)

model.fit(
    X_train_manual_scaled,
    y_train
)

LogisticRegression(C=0.01, max_iter=2000)

In [ ]:
proba = model.predict_proba(
    X_val_manual_scaled
)[:, 1]

score = gini(y_val, proba)
print("Gini (manual feature selection):", score)

Gini (manual feature selection): 0.4553509869005725


In [ ]:
l1_model = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=1.0,
    max_iter=2000
)

l1_model.fit(
    train_scaled,
    y_train
)

LogisticRegression(max_iter=2000, penalty='l1', solver='liblinear')

In [ ]:
coef = pd.Series(
    l1_model.coef_[0],
    index=X_train_final.columns
)

selected_features = coef[
    coef != 0
].index

X_train_l1 = X_train_final[selected_features]
X_val_l1 = X_val_final[selected_features]

ss_l1 = StandardScaler()

X_train_l1_scaled = ss_l1.fit_transform(X_train_l1)
X_val_l1_scaled = ss_l1.transform(X_val_l1)

In [ ]:
final_model = LogisticRegression(
    C=0.01,
    max_iter=2000
)

final_model.fit(
    X_train_l1_scaled,
    y_train
)

LogisticRegression(C=0.01, max_iter=2000)

In [ ]:
proba = final_model.predict_proba(
    X_val_l1_scaled
)[:, 1]

score = gini(y_val, proba)
print("Gini (manual feature selection):", score)

Gini (manual feature selection): 0.4564104692112376


разница малая, но l1 чуть лучше

In [ ]:
models = {
    "logreg": LogisticRegression(C=0.01, max_iter=2000),
}

def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1

for name, model in models.items():
    model.fit(train_scaled, y_train)

    proba = model.predict_proba(test_scaled)[:, 1]
    score = gini(y_test, proba)

    print(name, score)

logreg 0.4891747595638103


In [ ]:
models = {
    "logreg": LogisticRegression(C=0.01, max_iter=2000),
}

def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1

for name, model in models.items():
    model.fit(train_scaled, y_train)

    proba = model.predict_proba(train_scaled)[:, 1]
    score = gini(y_train, proba)

    print(name, score)

logreg 0.5171473003185167


эта лучше всех

мы хотим не купить плохую машину, поэтому нужно использовать Recall, F1-score

In [ ]:
def precision(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return tp / (tp + fp + 1e-12)

def recall(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn + 1e-12)

def f1(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * p * r / (p + r + 1e-12)

import numpy as np

def pr_auc(y_true, y_score):

    order = np.argsort(-y_score)

    y_true = np.array(y_true)[order]

    tp = 0
    fp = 0

    P = np.sum(y_true == 1)

    precisions = []
    recalls = []

    for i in range(len(y_true)):

        if y_true[i] == 1:
            tp += 1
        else:
            fp += 1

        precision = tp / (tp + fp + 1e-12)
        recall = tp / (P + 1e-12)

        precisions.append(precision)
        recalls.append(recall)

    return np.trapz(precisions[::-1], recalls[::-1])

In [ ]:
models = {
    "logreg": LogisticRegression(C=0.005, max_iter=1000),
    "gnb": GaussianNB(),
    "knn": KNeighborsClassifier(n_neighbors=5)
}

for name, model in models.items():
    model.fit(train_scaled, y_train)

    proba = model.predict(test_scaled)
    score = recall(y_test, proba)

    print(name, score)

logreg 0.18635502210991783
gnb 0.9617814276689827
knn 0.11876184459886288
